# NB06: SHAP analysis on the canonical opx-liq models (Phase 6R)

Reads the Phase 3R winning configuration JSON and loads the canonical RF
opx-liq T and P models built on the dynamically chosen feature set. No
feature engineering method is hard-coded.

## Purpose / Inputs / Outputs / Canonical decisions

**Purpose:** SHAP analysis on the canonical opx-liq RF models. Beeswarm plots, dependence plots for top features, and a feature-importance ranking used to verify that the model learned geochemically plausible signals.

**Inputs:** canonical opx-liq RF models, `WIN_FEAT`, `data/processed/opx_clean_opx_liq.parquet`.

**Outputs:** `results/nb06_shap_feature_importance.csv`, `figures/fig07_shap_*`, `figures/fig08_shap_*`, `figures/fig09_shap_*`.

**Canonical decisions:** Does not set canonical choices. Results are descriptive. If a SHAP-based feature pruning is adopted, it must be re-routed through NB03's feature-set winner selection.


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
from config import (
    ROOT, DATA_RAW, DATA_EXTERNAL, DATA_PROC, DATA_SPLITS, DATA_NATURAL,
    MODELS, FIGURES, RESULTS, LOGS,
    EXPETDB, LEPR_XLSX, LIN2023_NATURAL,
    FE3_FET_RATIO, KD_FEMG_MIN, KD_FEMG_MAX, WO_MAX_MOL_PCT,
    P_CEILING_KBAR, CATION_SUM_MIN, CATION_SUM_MAX,
    OXIDE_TOTAL_MIN, OXIDE_TOTAL_MAX,
    SEED_SPLIT, SEED_MODEL, SEED_NOISE_AUG, SEED_KMEANS,
    OPX_RAW_OXIDES, OPX_FULL_OXIDES, LIQ_OXIDES,
)
from src.plot_style import load_winning_config, canonical_model_filename
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import shap

In [ ]:
# Canonical features and prediction helpers from src/ (one source of truth).
from src.features import (
    build_feature_matrix,
    make_raw_features,
    make_alr_features,
    make_pwlr_features,
    augment_dataframe,
)
from src.models import predict_median, predict_iqr


In [ ]:
# Load canonical RF opx-liq T and P models (Phase 3R global winner)
# v7 Part H: per-family canonical spec replaces the legacy
# legacy pre-v7 single-winner assumption. Forest / T_C / opx_liq is the
# primary single-feature-set stand-in for notebooks that have not
# been restructured into per-family output blocks.
from src.data import canonical_model_spec
_spec_primary = canonical_model_spec('T_C', 'opx_liq', 'forest', RESULTS)
WIN_FEAT = _spec_primary['feature_set']
print(f'v7 primary (forest / T_C / opx_liq) feature set: {WIN_FEAT}')
model_T = joblib.load(MODELS / canonical_model_filename('T_C', 'opx_liq', 'forest', RESULTS))
model_P = joblib.load(MODELS / canonical_model_filename('P_kbar', 'opx_liq', 'forest', RESULTS))

df_liq = pd.read_parquet(DATA_PROC / 'opx_clean_opx_liq.parquet')
test_idx = np.load(DATA_SPLITS / 'test_indices_opx_liq.npy')
df_test = df_liq.loc[test_idx].copy()
print(f'Test set: {len(df_test)} experiments')

# T and P share the same canonical feature set under dynamic selection
X_test, feature_names = build_feature_matrix(df_test, WIN_FEAT, use_liq=True)
print(f'X_test shape: {X_test.shape}, n_features: {len(feature_names)}')

In [ ]:
# SHAP TreeExplainer for the pressure model
explainer_P = shap.TreeExplainer(model_P)
shap_values_P = explainer_P.shap_values(X_test, check_additivity=False)
print(f'shap_values_P shape: {np.asarray(shap_values_P).shape}')

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values_P, X_test, feature_names=feature_names, show=False)
plt.tight_layout()
plt.savefig(FIGURES / 'fig07_shap_P_beeswarm.png', dpi=300, bbox_inches='tight')
plt.savefig(FIGURES / 'fig_nb06_shap_P_beeswarm.png', dpi=300, bbox_inches='tight')  # v8-fix: canonical stem
plt.close()

In [ ]:
# SHAP TreeExplainer for the temperature model
explainer_T = shap.TreeExplainer(model_T)
shap_values_T = explainer_T.shap_values(X_test, check_additivity=False)
print(f'shap_values_T shape: {np.asarray(shap_values_T).shape}')

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values_T, X_test, feature_names=feature_names, show=False)
plt.tight_layout()
plt.savefig(FIGURES / 'fig08_shap_T_beeswarm.png', dpi=300, bbox_inches='tight')
plt.savefig(FIGURES / 'fig_nb06_shap_T_beeswarm.png', dpi=300, bbox_inches='tight')  # v8-fix: canonical stem
plt.close()

In [ ]:
# Dependence plots for top-3 features (P and T)
def top_indices(shap_values, k=3):
    sv = np.asarray(shap_values)
    if sv.ndim == 3:
        sv = sv[..., 0]
    importances = np.abs(sv).mean(axis=0)
    return np.argsort(importances)[-k:][::-1], importances

top_P_idx, imp_P_arr = top_indices(shap_values_P, k=3)
top_T_idx, imp_T_arr = top_indices(shap_values_T, k=3)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, feat_idx in zip(axes, top_P_idx):
    shap.dependence_plot(int(feat_idx), shap_values_P, X_test,
                         feature_names=feature_names, ax=ax, show=False)
plt.tight_layout()
plt.savefig(FIGURES / 'fig09_shap_P_dependence.png', dpi=300, bbox_inches='tight')
plt.close()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, feat_idx in zip(axes, top_T_idx):
    shap.dependence_plot(int(feat_idx), shap_values_T, X_test,
                         feature_names=feature_names, ax=ax, show=False)
plt.tight_layout()
plt.savefig(FIGURES / 'fig09_shap_T_dependence.png', dpi=300, bbox_inches='tight')
plt.close()

In [ ]:
# Save importance rankings (columns include feature set so downstream
# notebooks can confirm they are reading values from the right method)
sv_P = np.asarray(shap_values_P)
if sv_P.ndim == 3:
    sv_P = sv_P[..., 0]
sv_T = np.asarray(shap_values_T)
if sv_T.ndim == 3:
    sv_T = sv_T[..., 0]

imp_P = np.abs(sv_P).mean(axis=0)
imp_T = np.abs(sv_T).mean(axis=0)

imp_df = pd.DataFrame({
    'feature': feature_names,
    f'importance_P_{WIN_FEAT}': imp_P,
    f'importance_T_{WIN_FEAT}': imp_T,
    'feature_set': WIN_FEAT,
})
imp_df.to_csv(RESULTS / 'nb06_shap_feature_importance.csv', index=False)

top10 = imp_df.sort_values(f'importance_P_{WIN_FEAT}', ascending=False).head(10)
top10.to_csv(RESULTS / 'table5_shap_importance.csv', index=False)

print('=== NB06 complete ===')
print(f'  feature set: {WIN_FEAT}')
print(f'  n features: {len(feature_names)}')

## Robustness checks (merged from deprecated nb06b)

Six diagnostic tests rerun on the canonical opx_liq pressure model (using the
winning feature set from Phase 3R). Each test is a targeted probe of whether
the model is learning real physicochemical signal versus exploiting a
statistical proxy of the laboratory experimental design.


In [ ]:
# Load the canonical train + test feature matrices for robustness testing.
train_idx = np.load(DATA_SPLITS / 'train_indices_opx_liq.npy')
df_train = df_liq.loc[train_idx].copy()

X_train, _ = build_feature_matrix(df_train, WIN_FEAT, use_liq=True)
y_train_P = df_train['P_kbar'].values
y_train_T = df_train['T_C'].values
y_test_P = df_test['P_kbar'].values
y_test_T = df_test['T_C'].values

# X_train/X_test are numpy arrays; wrap in DataFrames so we can drop columns by
# feature name in the ablation test below.
X_train_df = pd.DataFrame(X_train, columns=feature_names)
X_test_df = pd.DataFrame(X_test, columns=feature_names)

from sklearn.base import clone
from sklearn.metrics import mean_squared_error, r2_score

# Refit the canonical models once and record original RMSE.
model_P_fit = clone(model_P); model_P_fit.fit(X_train_df.values, y_train_P)
model_T_fit = clone(model_T); model_T_fit.fit(X_train_df.values, y_train_T)
original_rmse_P = float(np.sqrt(mean_squared_error(
    y_test_P, predict_median(model_P_fit, X_test_df.values))))
original_rmse_T = float(np.sqrt(mean_squared_error(
    y_test_T, predict_median(model_T_fit, X_test_df.values))))
print(f'Baseline RMSE P = {original_rmse_P:.3f} kbar | T = {original_rmse_T:.2f} C')


### Robustness A. Ablation test

Drop every feature whose name contains `liq_SiO2` or `liq_MgO` (these dominate
SHAP importance) and refit the pressure and temperature models to see how
much performance degrades when the candidate proxies are removed.


In [ ]:
suspect_substr = ['liq_SiO2', 'liq_MgO']
suspect_cols = [c for c in feature_names
                if any(s in c for s in suspect_substr)]
print(f'Dropping {len(suspect_cols)} features: {suspect_cols}')

X_tr_ab = X_train_df.drop(columns=suspect_cols)
X_te_ab = X_test_df.drop(columns=suspect_cols)

model_ab_P = clone(model_P); model_ab_P.fit(X_tr_ab.values, y_train_P)
model_ab_T = clone(model_T); model_ab_T.fit(X_tr_ab.values, y_train_T)
rmse_ab_P = float(np.sqrt(mean_squared_error(
    y_test_P, predict_median(model_ab_P, X_te_ab.values))))
rmse_ab_T = float(np.sqrt(mean_squared_error(
    y_test_T, predict_median(model_ab_T, X_te_ab.values))))

print(f'P RMSE: {original_rmse_P:.3f} -> {rmse_ab_P:.3f} kbar '
      f'(delta = {rmse_ab_P - original_rmse_P:+.3f})')
print(f'T RMSE: {original_rmse_T:.2f} -> {rmse_ab_T:.2f} C    '
      f'(delta = {rmse_ab_T - original_rmse_T:+.2f})')


### Robustness B. Experimental proxy check

If liquid SiO2 and MgO are simply acting as proxies for the experimental P
and T set points (because different labs run different ranges), the raw
scatterplots should show strong monotonic trends.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(df_liq['P_kbar'], df_liq['liq_SiO2'],
                alpha=0.5, c='#2c7bb6', edgecolor='k', linewidth=0.5)
axes[0].set_xlabel('Pressure (kbar)')
axes[0].set_ylabel('Liquid SiO2 (wt%)')
axes[0].set_title('liq_SiO2 vs. Pressure')

axes[1].scatter(df_liq['T_C'], df_liq['liq_MgO'],
                alpha=0.5, c='#d7191c', edgecolor='k', linewidth=0.5)
axes[1].set_xlabel('Temperature (C)')
axes[1].set_ylabel('Liquid MgO (wt%)')
axes[1].set_title('liq_MgO vs. Temperature')

plt.tight_layout()
plt.savefig(FIGURES / 'fig_nb06_proxy_check.png', dpi=300, bbox_inches='tight')
plt.show()


### Robustness C. Feature correlation heatmap

Spearman-style visual check among the leading feature candidates and the
targets. Strong cross-correlation between `liq_*` oxides and P or T would
support the proxy concern.


In [ ]:
import seaborn as sns

candidate_cols = [c for c in
                  ['Al_VI', 'CaO', 'liq_SiO2', 'liq_MgO', 'P_kbar', 'T_C']
                  if c in df_liq.columns]
if len(candidate_cols) < 4:
    print(f'Skipping correlation check: only {candidate_cols} present')
else:
    corr = df_liq[candidate_cols].corr()
    fig, ax = plt.subplots(figsize=(7, 6))
    sns.heatmap(corr, annot=True, cmap='RdBu_r', vmin=-1, vmax=1,
                fmt='.2f', linewidths=0.5, ax=ax)
    ax.set_title('Correlation: candidate dominators vs. targets')
    plt.tight_layout()
    plt.savefig(FIGURES / 'fig_nb06_correlation_check.png',
                dpi=300, bbox_inches='tight')
    plt.show()


### Robustness D. Y-randomization (permutation) test

Shuffle the training labels to destroy the physical relationship, refit with
identical hyperparameters, and predict the real test set. A healthy model
collapses to `R2 <= 0` here; leakage would keep `R2 > 0`.


In [ ]:
rng = np.random.default_rng(SEED_SPLIT)
y_train_P_shuffled = rng.permutation(y_train_P)

model_sanity = clone(model_P)
model_sanity.fit(X_train_df.values, y_train_P_shuffled)
pred_shuffled = predict_median(model_sanity, X_test_df.values)
rmse_shuffled = float(np.sqrt(mean_squared_error(y_test_P, pred_shuffled)))
r2_shuffled = float(r2_score(y_test_P, pred_shuffled))

print(f'Baseline P RMSE:   {original_rmse_P:.3f} kbar')
print(f'Shuffled P RMSE:   {rmse_shuffled:.3f} kbar')
print(f'Shuffled R^2:      {r2_shuffled:+.3f}   (expected <= 0)')


### Robustness E. Dummy regressor baseline

Constant-mean predictor. Any nontrivial model must beat this.


In [ ]:
from sklearn.dummy import DummyRegressor

dummy_model = DummyRegressor(strategy='mean')
dummy_model.fit(X_train_df.values, y_train_P)
rmse_dummy = float(np.sqrt(mean_squared_error(
    y_test_P, dummy_model.predict(X_test_df.values))))
print(f'RF P RMSE:    {original_rmse_P:.3f} kbar')
print(f'Dummy P RMSE: {rmse_dummy:.3f} kbar')


### Robustness F. Perfect-signal injection

Inject a fake feature that equals `y_P + small_noise`. Under the canonical
constrained hyperparameters, a single perfect feature may not dominate
because `max_features` controls the split pool. The unconstrained variant
forces the model to consider the fake feature every split, and a linear
sanity check confirms the fake feature is present in the data.


In [ ]:
# Constrained: injects fake feature, uses canonical hyperparameters.
X_tr_fake = X_train_df.copy()
X_te_fake = X_test_df.copy()
X_tr_fake['FAKE_PERFECT_SIGNAL'] = (y_train_P
                                    + rng.normal(0, 0.1, len(y_train_P)))
X_te_fake['FAKE_PERFECT_SIGNAL'] = (y_test_P
                                    + rng.normal(0, 0.1, len(y_test_P)))

model_fake = clone(model_P)
model_fake.fit(X_tr_fake.values, y_train_P)
rmse_fake = float(np.sqrt(mean_squared_error(
    y_test_P, predict_median(model_fake, X_te_fake.values))))
print(f'Constrained perfect-signal P RMSE: {rmse_fake:.3f} kbar '
      f'(baseline {original_rmse_P:.3f})')

# Unconstrained: max_features=None forces the fake feature to be seen every split.
from sklearn.ensemble import RandomForestRegressor
model_fake_unc = RandomForestRegressor(
    n_estimators=200, max_features=None,
    random_state=SEED_MODEL, n_jobs=-1,
)
model_fake_unc.fit(X_tr_fake.values, y_train_P)
rmse_fake_unc = float(np.sqrt(mean_squared_error(
    y_test_P, predict_median(model_fake_unc, X_te_fake.values))))
print(f'Unconstrained perfect-signal P RMSE: {rmse_fake_unc:.3f} kbar '
      f'(expected ~0.1)')

# Linear sanity check: a plain OLS with the fake feature included should nail it.
from sklearn.linear_model import LinearRegression
linear_model = LinearRegression()
linear_model.fit(X_tr_fake.fillna(0).values, y_train_P)
rmse_linear = float(np.sqrt(mean_squared_error(
    y_test_P, linear_model.predict(X_te_fake.fillna(0).values))))
print(f'Linear perfect-signal P RMSE: {rmse_linear:.3f} kbar')
